In [3]:
import os

cwd = os.getcwd()

print(cwd)

/Users/ettyekhon/code/src/petroineos/petroineos-related/taxi-data/06-batch/code


In [10]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [11]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [12]:
spark.version

'4.1.1'

In [13]:
!ls -lh fhvhv_tripdata_2021-02.parquet

-rw-r--r--@ 1 ettyekhon  staff   289M 30 Jun  2022 fhvhv_tripdata_2021-02.parquet


In [14]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [15]:
!wget -O fhvhv_tripdata_2021-02.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-02.parquet

--2026-03-07 16:54:21--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-02.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:f800:b:20a5:b140:21, 2600:9000:2684:2e00:b:20a5:b140:21, 2600:9000:2684:fa00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:f800:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 302633211 (289M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-02.parquet’

fhvhv_tripdata_2021 100%[===================>] 288.61M  81.5MB/s    in 3.5s    

2026-03-07 16:54:25 (81.5 MB/s) - ‘fhvhv_tripdata_2021-02.parquet’ saved [302633211/302633211]



In [21]:
df = spark.read.parquet('fhvhv_tripdata_2021-02.parquet') \
    .select('hvfhs_license_num', 'dispatching_base_num',
            'pickup_datetime', 'dropoff_datetime',
            'PULocationID', 'DOLocationID')

df = df.repartition(24)

df.write.parquet('data/pq/fhvhv/2021/02/', mode='overwrite')

26/03/07 16:55:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/07 16:55:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/07 16:55:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/07 16:55:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/07 16:55:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/07 16:55:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/07 16:55:03 WARN MemoryManager: Total allocation exceeds 95.

In [22]:
df = spark.read.parquet('data/pq/fhvhv/2021/02/')

**Q3**: How many taxi trips were there on February 15?

In [23]:
from pyspark.sql import functions as F

In [24]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)).filter("pickup_date = '2021-02-15'").count()

367170

In [25]:
df.createOrReplaceTempView('fhvhv_2021_02')

In [26]:
spark.sql("""
SELECT
    COUNT(1)
FROM 
    fhvhv_2021_02
WHERE
    to_date(pickup_datetime) = '2021-02-15';
""").show()

+--------+
|count(1)|
+--------+
|  367170|
+--------+



**Q4**: Longest trip for each day

In [27]:
df.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID']

In [29]:
df \
    .withColumn('duration', F.unix_timestamp('dropoff_datetime') - F.unix_timestamp('pickup_datetime')) \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .groupBy('pickup_date') \
        .max('duration') \
    .orderBy('max(duration)', ascending=False) \
    .limit(5) \
    .show()

[Stage 22:==============>                                          (3 + 9) / 12]

+-----------+-------------+
|pickup_date|max(duration)|
+-----------+-------------+
| 2021-02-11|        75540|
| 2021-02-17|        57221|
| 2021-02-20|        44039|
| 2021-02-03|        40653|
| 2021-02-19|        37577|
+-----------+-------------+



In [31]:
spark.sql("""
SELECT
    to_date(pickup_datetime) AS pickup_date,
    MAX((unix_timestamp(dropoff_datetime) - unix_timestamp(pickup_datetime)) / 60) AS duration
FROM 
    fhvhv_2021_02
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 10;
""").show()

[Stage 25:>                                                       (0 + 12) / 12]

+-----------+-----------------+
|pickup_date|         duration|
+-----------+-----------------+
| 2021-02-11|           1259.0|
| 2021-02-17|953.6833333333333|
| 2021-02-20|733.9833333333333|
| 2021-02-03|           677.55|
| 2021-02-19|626.2833333333333|
| 2021-02-25|            583.5|
| 2021-02-18|576.8666666666667|
| 2021-02-10|569.4833333333333|
| 2021-02-21|           537.05|
| 2021-02-09|534.7833333333333|
+-----------+-----------------+



**Q5**: Most frequent `dispatching_base_num`

How many stages this spark job has?



In [32]:
spark.sql("""
SELECT
    dispatching_base_num,
    COUNT(1)
FROM 
    fhvhv_2021_02
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show()

+--------------------+--------+
|dispatching_base_num|count(1)|
+--------------------+--------+
|              B02510| 3233664|
|              B02764|  965568|
|              B02872|  882689|
|              B02875|  685390|
|              B02765|  559768|
+--------------------+--------+



In [33]:
df \
    .groupBy('dispatching_base_num') \
        .count() \
    .orderBy('count', ascending=False) \
    .limit(5) \
    .show()

+--------------------+-------+
|dispatching_base_num|  count|
+--------------------+-------+
|              B02510|3233664|
|              B02764| 965568|
|              B02872| 882689|
|              B02875| 685390|
|              B02765| 559768|
+--------------------+-------+



**Q6**: Most common locations pair

In [34]:
df_zones = spark.read.parquet('zones')

In [35]:
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [36]:
df.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID']

In [37]:
df_zones.createOrReplaceTempView('zones')

In [38]:
spark.sql("""
SELECT
    CONCAT(pul.Zone, ' / ', dol.Zone) AS pu_do_pair,
    COUNT(1)
FROM 
    fhvhv_2021_02 fhv LEFT JOIN zones pul ON fhv.PULocationID = pul.LocationID
                      LEFT JOIN zones dol ON fhv.DOLocationID = dol.LocationID
GROUP BY 
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show()

[Stage 36:>                                                       (0 + 12) / 12]

+--------------------+--------+
|          pu_do_pair|count(1)|
+--------------------+--------+
|East New York / E...|   45041|
|Borough Park / Bo...|   37329|
| Canarsie / Canarsie|   28026|
|Crown Heights Nor...|   25976|
|Bay Ridge / Bay R...|   17934|
+--------------------+--------+



In [39]:
spark.stop()